---
title: Query via Icechunk
---

# Query NISAR GCOV via Icechunk

Open the virtual Icechunk store created in the previous notebook and
query 10 random points. Only the chunks containing those points are
fetched from NASA's S3 storage.

In [ ]:
import warnings

warnings.filterwarnings("ignore", message="Numcodecs codecs are not in the Zarr")
warnings.filterwarnings(
    "ignore", category=UserWarning, message=".*does not have a Zarr V3 specification.*"
)
warnings.filterwarnings("ignore", category=FutureWarning, module="earthaccess")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="tqdm")

## Open the Icechunk store

In [ ]:
import earthaccess
import icechunk
import xarray as xr

earthaccess.login()

storage = icechunk.local_filesystem_storage("nisar-icechunk")
config = icechunk.RepositoryConfig.default()

# Configure access to the original data on NASA S3
bucket = "s3://nisar-prod-asf-cumulus-prod-protected"
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=bucket,
        store=icechunk.s3_store(region="us-west-2"),
    ),
)
virtual_credentials = icechunk.containers_credentials(
    {bucket: icechunk.s3_credentials_from_env()}
)

repo = icechunk.Repository.open(
    storage=storage,
    config=config,
    authorize_virtual_chunk_access=virtual_credentials,
)

session = repo.readonly_session()
dt = xr.open_datatree(session.store, engine="zarr", consolidated=False, zarr_format=3)
dt

## Select the GCOV data and query 10 random points

In [ ]:
import numpy as np
import time

# Get the main data group
gcov = dt["science/LSAR/GCOV/grids/frequencyA"].dataset

rng = np.random.default_rng(42)
ny, nx = gcov.dims["y"], gcov.dims["x"]
yi = rng.integers(0, ny, size=10)
xi = rng.integers(0, nx, size=10)

# Read HHHH (HH-HH covariance) at the 10 points
t0 = time.perf_counter()
values = gcov["HHHH"].values[yi, xi]
elapsed = time.perf_counter() - t0

print(f"Queried 10 points in {elapsed:.2f}s")

## Results

In [ ]:
import pandas as pd

results = pd.DataFrame(
    {
        "y_index": yi,
        "x_index": xi,
        "HHHH": values,
    }
)
results

## Visualize

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 8))
data = gcov["HHHH"].values
im = ax.imshow(data, cmap="viridis", aspect="equal")
ax.scatter(xi, yi, c="red", s=40, edgecolors="white", linewidths=0.5, zorder=5)
ax.set_title("NISAR GCOV HHHH covariance")
plt.colorbar(im, ax=ax, shrink=0.6)
fig.tight_layout()